# Attention and the Transformer Block

> In the previous section, we added positional information to each token's vector. This completes the input layer: each token now carries both semantic information ("what this word is") and positional information ("where this word sits in the sequence"). But there is a problem—these vectors are computed independently, with no communication between them. In other words, the model does not yet know how tokens relate to each other.
>
> This section starts from that problem and introduces the Attention mechanism, which lets each token extract information from its context on demand. We will first walk through the Attention computation by hand, then progressively add causal masking, multi-head attention, a feed-forward network, residual connections, and LayerNorm, finally assembling everything into a Transformer Block.

At the end of the previous section, each token had a vector containing both semantics and position. Take "the cat sat on the mat" as an example: the sentence is first split by the Tokenizer into a token sequence [the, cat, sat, on, the, mat], then each token is looked up in the Embedding table to get a vector, and finally positional encoding is added. Each vector carries two pieces of information: what the word is, and where it sits in the sentence. This is already enough in some respects—the model can distinguish "the" at different positions. But there is a critical gap: these vectors are independent of each other. The vector for "sat" only contains information about "sat" itself; the vector for "cat" only contains information about "cat". All other tokens in the context are invisible to each token.

A great deal of information in language comes from context. For "sat" to understand its own meaning, it needs to know that "cat" precedes it (who is sitting) and "mat" follows (where it is sitting). The word "bank" in "river bank" and "bank account" consists of the same four letters, yet carries completely different meanings—the only way to distinguish them is by the surrounding words. These examples show that relying solely on each token's own semantics and position is not enough; the model also needs a way for tokens to exchange information.

The overall Transformer architecture is therefore divided into three stages:

```
Input layer:   Tokenizer → Embedding → Position Encoding    ← previous three sections completed
Core layer:    N Transformer Blocks                             ← this section's focus
Output layer:  Linear → logits                                ← next section
```

This section builds the middle piece—the Transformer Block. The Block's core mechanism is called Attention: each token first determines which parts of the context are more relevant, then mixes information weighted by that relevance.

## Key Questions

After studying this section, you should be able to answer:

1. What problem does Attention solve?
2. What do Q, K, and V each represent?
3. What are the four steps of Scaled Dot-Product Attention?
4. Why does GPT need a Causal Mask?
5. How are Multi-Head Attention and the Transformer Block assembled?

## 1. Intuition Behind Attention

Attention output can be described as a set of weights. Suppose the model is processing "the cat sat on the mat". When it processes "sat", the attention weights might look like this:

```
      the  cat  sat  on  the  mat
sat: 0.05 0.35 0.10 0.05 0.05 0.40
```

These numbers sum to 1. "cat" and "mat" have large weights, meaning "sat" reads more information from them. The resulting new vector for "sat" then blends in the context of "a cat sitting on a mat".

Computing these attention weights is the core problem Attention solves. It introduces three vectors—Q (Query), K (Key), and V (Value)—playing three roles: "asking a question", "providing a label", and "offering content":

- Q (Query): What am I looking for? For instance, "sat" might be asking: "Who is performing this action? Where is the action taking place?"
- K (Key): What label do I carry? For instance, "cat"'s Key might signal to others: "I am the doer of an action."
- V (Value): What content can I provide? If "cat" is attended to, what actually gets mixed into the output is its Value.

A helpful analogy is looking up references: Query is the question you want to answer, Key is the title or tag of each document, and Value is the body of the document. You first use Query and Key to determine which documents are relevant, then read in the Value of the relevant documents in proportion.

In Self-Attention, Q, K, and V are all computed from the same input matrix X, but multiplied by three different weight matrices:

```
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

The same input goes through three different linear projections and takes on three different roles. Let's construct X in code and compute Attention step by step.

In [ ]:
# === Vocabulary → Sentence → Token IDs → Embedding lookup → X ===
import torch
import torch.nn as nn
_ = torch.manual_seed(42)
vocab = {"the": 0, "a": 1, "cat": 2, "dog": 3, "sat": 4,
         "ran": 5, "on": 6, "mat": 7, "[PAD]": 8, "[UNK]": 9}
vocab_size = len(vocab)
id2word = {v: k for k, v in vocab.items()}

# Sentence → token ids
sentence = ["the", "cat", "sat"]
token_ids = [vocab[w] for w in sentence]  # [0, 2, 4]

# Token ids → Embedding lookup → X
d_model = 4
embedding = nn.Embedding(vocab_size, d_model)
token_ids_tensor = torch.tensor(token_ids)  # [3]
X = embedding(token_ids_tensor)             # [3, 4]

print(f"Vocabulary size: {vocab_size}, Sentence: {' '.join(sentence)}")
print(f"Token IDs: {token_ids}")
print(f"X shape: {list(X.shape)}  ← [seq_len=3, d_model=4]")
print(f"\nX = (from Embedding lookup, not randn):\n{X}")
print(f"\nExplanation: X[0]='the' → {X[0].tolist()}")
print(f"      X[1]='cat' → {X[1].tolist()}")
print(f"      X[2]='sat' → {X[2].tolist()}")

In [ ]:
# === Starting from X above, compute Q/K/V ===
import torch.nn as nn
seq_len = X.shape[0]  # = 3
d_k = 4

# Q/K/V all come from X, but each is multiplied by a different weight matrix
# (using nn.Linear for consistency with the MHA implementation later)
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)  # [3, 4] — each token's "query"
K = W_K(X)  # [3, 4] — each token's "label"
V = W_V(X)  # [3, 4] — each token's "content"

print(f"X shape: {X.shape}  →  Q/K/V shape: {Q.shape}")
print(f"→ Q, K, V all come from the same X, multiplied by different matrices")

## 2. Scaled Dot-Product Attention

Attention computation has four steps. The first step uses Q and K to compute relevance scores—row i, column j represents how interested token i is in token j. A larger dot product means a stronger match.

In [ ]:
# Step 1: Attention scores = Q × K^T
# Row i, column j = raw relevance of token i toward token j
attention_scores = Q @ K.T  # [3, 4] @ [4, 3] = [3, 3]

print(f"Attention score matrix {list(attention_scores.shape)} = [{seq_len}×{seq_len}]:")
print(attention_scores)
print(f"\nRow i = scores of token {list(range(seq_len))} toward each token")

**Scaling**

Dot products can be large. When values are too large, softmax becomes overconfident and training becomes unstable.

To address this, we divide by `\u221ad_k` to keep the scores stable.

Why \u221ad_k and not some other number? This can be understood from the perspective of variance. Suppose each element of Q and K is an independent random variable with mean 0 and variance 1. Then $q_i k_i$ also has variance 1, and the dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$.

In other words, the higher the dimension, the larger the typical absolute value of the dot product. When $d_k = 64$, the typical magnitude of the dot product is around 8 ($\sqrt{64} = 8$), and softmax inputs near \u00b18 are already close to saturation, where gradients become very small.

Dividing by $\sqrt{d_k}$ effectively pulls the variance of the dot product back to 1, keeping softmax in a region with sufficient gradients.

In [ ]:
# Step 2: Scale by / √d_k → prevent large dot products when d_k is big, which causes softmax gradient vanishing
import math
d_k = Q.shape[-1]
scaled_scores = attention_scores / math.sqrt(d_k)

print(f"Scaling factor: √{d_k} = {math.sqrt(d_k):.2f}")
print(f"Before scaling token 0: {attention_scores[0].tolist()}")
print(f"After scaling token 0:  {scaled_scores[0].tolist()}  ← values shrink, relative order preserved")

The third step applies softmax to turn each row of scores into probabilities—each row sums to 1, representing how much attention this token pays to each other token.

The fourth step uses these weights to mix V. Whoever has a larger weight contributes more information. This way, each token's output blends in the context it attended to.

In [ ]:
# Step 3: Softmax → turn scores into probabilities (each row sums to 1)
import torch.nn.functional as F
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Attention weight matrix {list(attention_weights.shape)}:")
print(attention_weights)

# Verify each row sums to 1
print(f"\nRow sums: {attention_weights.sum(dim=-1).tolist()}  ← all 1.0")

In [ ]:
# Step 4: Weighted sum — mix V using attention weights
output = attention_weights @ V  # [3, 3] @ [3, 4] = [3, 4]

print(f"Output shape: {list(output.shape)} = [{seq_len}, {d_model}]")
print(f"\nInput  token 0: {X[0].tolist()}")
print(f"Output token 0: {output[0].tolist()}")
print(f"→ Different! Because token 0 blended in information from tokens 1 and 2")

### Four-Step Summary

```
1. Q = X @ W_Q, K = X @ W_K, V = X @ W_V
2. scores = Q @ K^T
3. weights = softmax(scores / √d_k)
4. output = weights @ V
```

This is the complete computation of Scaled Dot-Product Attention.

## 3. Causal Masking

GPT is a generative model. When predicting the next token, it must not see future tokens. When predicting the word after "cat", the model must not peek at the answer "sat"—otherwise training would be like looking at the answer key during an exam: the loss looks good but the model never learns to generate.

The solution is to set future position scores to $-\infty$ before softmax. After softmax, positions corresponding to $-\infty$ become 0, which is equivalent to being completely masked:

```
[a, b, c]  →  [a, -∞, -∞]
[d, e, f]  →  [d,  e, -∞]
[g, h, i]  →  [g,  h,  i]
```

In [ ]:
# Causal mask: lower triangular matrix, 1=allowed, 0=masked (→ -inf)
import torch
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len))

mask = causal_mask(5)
print("Causal Mask (1=allowed, 0=masked):")
print(mask)
# Row i = positions token i can see: token 0 only sees [0], token 2 sees [0,1,2]

## 4. Multi-Head Attention

Single-head Attention has only one perspective. The same sentence can be viewed from different angles: one head might focus on syntactic relationships, another on positional distance, and a third on semantic relationships. Multiple heads working in parallel prevent the model from fixating on just one type of relationship.

Each head independently computes Attention (with its own Q/K/V projections), then the results from all heads are concatenated and passed through a linear transformation to produce the final output. GPT-2 has 12 heads; GPT-3 has 96.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention; with a causal mask it becomes causal self-attention.

    Args:
        d_model: input/output dimension
        num_heads: number of attention heads
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Linear projections for Q, K, V (batch all heads into matrix ops)
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        # Output projection
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        """
        Input:  x shape = [batch, seq_len, d_model]
        Output:   shape = [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape

        # 1. Linear projections + split into heads
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Attention scores: Q @ K^T
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3. Mask (e.g., set future positions to -inf)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 4. Softmax
        weights = F.softmax(scores, dim=-1)

        # 5. Weighted sum
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]

        # 6. Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)

In [ ]:
# Test multi-head attention
import torch
d_model, num_heads = 8, 2
mha = MultiHeadAttention(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)                         # batch=1, 5 tokens, 8 dims
test_mask = causal_mask(5).unsqueeze(0).unsqueeze(0)        # [1, 1, 5, 5]

out = mha(test_x, test_mask)
print(f"Input: {test_x.shape}  →  Output: {out.shape}  ← shape unchanged, content blends visible context")

## 5. Assembling the Transformer Block

A Transformer Block consists of four components:

- **Attention** (Multi-Head Self-Attention): lets tokens see each other, solving the problem of context information flow
- **FFN** (Feed-Forward Network): applies a deeper transformation at each position independently, solving the problem of how to process information after reading context
- **Residual** (residual connections): `output = input + sublayer output`. Raw information has a direct path through, making deep networks easier to train
- **LayerNorm**: pulls each layer's values back to a stable range, preventing numerical instability as layers increase

The assembly order is:

```
x → Attention → Add & Norm → FFN → Add & Norm
```

This implementation uses the Post-LN variant: sublayer first, then residual addition and LayerNorm. Real LLMs often use Pre-LN or RMSNorm, which we will upgrade to later.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
class FeedForward(nn.Module):
    """FFN: two fully-connected layers, expand 4x then compress back"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class TransformerBlock(nn.Module):
    """A Transformer decoder layer: Attention + FFN, each with residual + LayerNorm"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Attention + residual + Norm
        x = self.norm2(x + self.ffn(x))              # FFN + residual + Norm
        return x

# Test
block = TransformerBlock(d_model=8, num_heads=2)
test_out = block(test_x, test_mask)
print(f"TransformerBlock Input: {test_x.shape}  →  Output: {test_out.shape}  ← shape unchanged")

## Summary

What we learned in this section:

- [ ] Embedding only gives each token an independent vector; tokens do not communicate
- [ ] Attention lets each token read information from context, weighted by relevance
- [ ] Q is the query (what am I looking for), K is the label (what am I), V is the content (what can I offer)
- [ ] Scaled Dot-Product Attention has four steps: compute scores → scale → softmax → weighted sum
- [ ] Causal Mask prevents GPT from peeking at the future—sets future position scores to -∞
- [ ] Multi-Head is multiple perspectives in parallel, concatenated at the end
- [ ] Transformer Block = Attention + FFN + Residual + LayerNorm

In the next section, we will stack Transformer Blocks to build a complete Mini-GPT.

## Appendix: RNN vs Transformer

This section answers a deeper question: why is Transformer better suited than RNN for long-range dependencies?

Rather than relying on slogans, we will directly run two PyTorch modules:

1. `torch.nn.RNN`
2. `torch.nn.MultiheadAttention`

Then we backpropagate from the last position and check whether gradients can reach earlier positions.

**The old approach: RNN passes messages step by step**

When RNN reads a sentence, it passes information forward one word at a time.

```
x1 → RNN → h1 → RNN → h2 → RNN → h3 → ...
```

`h` can be understood as "what I have remembered so far".

**Problem: long-range information is easily lost**

If the sentence is very long, information from the beginning must pass through many steps to reach the end. The further it travels, the more likely it is to be lost.

This is like the telephone game: after the first sentence is passed 100 times, it has usually become distorted.

**Seeing the problem through experiment first**

The code below demonstrates one thing: the longer the sequence, the smaller the gradient at the beginning positions.

A very small gradient means: the model has difficulty learning from distant earlier context.

**Experiment Step 1: Load PyTorch's built-in RNN**

We will not write the RNN formula by hand here. Instead, we use `torch.nn.RNN` directly. Focus on the experimental phenomenon: can the output from the last position propagate learning signals back to early positions?

In [ ]:
# Experiment Step 1: Load PyTorch's built-in RNN

import torch
import torch.nn as nn
torch.manual_seed(42)

seq_len = 80
batch_size = 1
input_dim = 4
hidden_dim = 8

rnn = nn.RNN(
    input_size=input_dim,
    hidden_size=hidden_dim,
    num_layers=1,
    nonlinearity="tanh",
    batch_first=True,
)

x = torch.randn(batch_size, seq_len, input_dim, requires_grad=True)

print("=== torch.nn.RNN ===")
print(rnn)
print(f"Input shape: {tuple(x.shape)} = [batch, seq_len, input_dim]")
print("This step only prepares the model and input. Next step: observe hidden states and gradients.")

**Experiment Step 2: Run one forward pass**

`nn.RNN` consumes the entire sequence at once.

It outputs the hidden state at every position, plus the final hidden state.

In [ ]:
# Experiment Step 2: RNN forward pass
outputs, h_last = rnn(x)

print("=== Forward pass ===")
print(f"outputs shape: {tuple(outputs.shape)}")
print(f"h_last shape:  {tuple(h_last.shape)}")
print()
print("outputs[:, t, :] is the hidden state at position t.")
print("h_last is the hidden state passed from the final step.")

**Experiment Step 3: Backpropagate from the last step**

We focus on one question: how sensitive is the last step's output to each input position?

The approach: turn the last hidden state into a loss, then call `backward()`.

First look at the gradient values at a few positions, then draw conclusions.

In [ ]:
# Experiment Step 3: Print input gradients
loss = outputs[:, -1, :].pow(2).mean()
loss.backward()

# Input gradient magnitude at each time position: [seq_len]
grad_by_pos = x.grad.norm(dim=-1).squeeze(0)

print("=== Gradients from last output ===")
print(f"loss from last output: {loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    print(f"position {pos:2d} gradient norm: {grad_by_pos[pos].item():.10f}")

first_grad = grad_by_pos[0].item()
middle_grad = grad_by_pos[seq_len // 2].item()
last_grad = grad_by_pos[-1].item()

print()
print("From this run's results:")
print(f"  Beginning position gradient: {first_grad:.3e}")
print(f"  Middle position gradient:    {middle_grad:.3e}")
print(f"  Last position gradient:       {last_grad:.3e}")
print()
print("Observation: gradients near the last step are largest, and they typically approach 0 further back.")
print("So what this shows is: the last step receives very weak learning signals from distant inputs.")

**Experiment Step 4: View the gradient curve**

The plot below shows the gradient magnitude at each position in the input sequence.

Look at the plot first, then read the conclusion: the curve does not smoothly "gradually decrease"; instead, most earlier positions are nearly flat at zero, with only positions near the end visibly rising.

In [ ]:
import matplotlib.pyplot as plt
# Experiment Step 4: RNN gradient distribution
import torch
positions = torch.arange(seq_len)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), linewidth=2)
plt.scatter(positions.numpy(), grad_by_pos.detach().numpy(), s=10)
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN input gradient by position")
plt.grid(True, alpha=0.3)
plt.show()

near_end = grad_by_pos[-5:].mean().item()
far_start = grad_by_pos[:5].mean().item()
print("From the corresponding numerical values:")
print(f"  Average gradient of first 5 positions: {far_start:.3e}")
print(f"  Average gradient of last 5 positions:  {near_end:.3e}")
print()
print("Observation: in this run, gradients are concentrated near the end of the sequence.")
print("This illustrates the common difficulty RNN faces with long-range dependencies: distant positions receive very little learning signal.")

**Experiment Step 5: What happens as the sequence gets longer?**

We keep the RNN configuration the same and only change the sequence length.

Run the curves first, then compare the leftmost and rightmost values of each curve. This plot uses a log scale—one tick mark on a log scale usually means an order-of-magnitude difference.

In [ ]:
import matplotlib.pyplot as plt
# Experiment Step 5: Gradient comparison under different sequence lengths
import torch
import torch.nn as nn
def run_rnn_with_length(length):
    """Use the same nn.RNN configuration, return input gradient magnitude per position"""
    torch.manual_seed(42)
    model = nn.RNN(input_dim, hidden_dim, nonlinearity="tanh", batch_first=True)
    sample = torch.randn(batch_size, length, input_dim, requires_grad=True)
    out, _ = model(sample)
    loss = out[:, -1, :].pow(2).mean()
    loss.backward()
    return sample.grad.norm(dim=-1).squeeze(0).detach()

lengths = [10, 20, 40, 80]
length_results = {}

plt.figure(figsize=(7, 4))
for length in lengths:
    grads = run_rnn_with_length(length)
    length_results[length] = grads
    relative_pos = torch.linspace(0, 1, steps=length)
    plt.plot(relative_pos.numpy(), grads.numpy(), label=f"seq_len={length}")

plt.xlabel("Relative sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN gradients under different sequence lengths")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("From this run's results:")
for length in lengths:
    grads = length_results[length]
    first = grads[0].item()
    last = grads[-1].item()
    ratio = last / max(first, 1e-30)
    print(f"  seq_len={length:2d}: first={first:.3e}, last={last:.3e}, last/first≈{ratio:.3e}")

print()
print("Observation: the longer the sequence, the more orders of magnitude smaller the earliest position's gradient becomes.")
print("Conclusion: RNN can process sequences, but long-range learning signals are harder to propagate back stably.")
print("The core motivation for Attention is: let distant tokens communicate in far fewer steps.")

**Experiment Step 6: Switch to PyTorch Attention and check gradients**

The RNN problem we just saw: learning signals from the last step must propagate step by step back to the beginning.

What about Attention? This time we directly use `torch.nn.MultiheadAttention`. Same question: backpropagate only from the last position and check whether earlier positions receive gradients.

We use a causal mask here, so the last position can see all earlier positions, but earlier positions cannot see the future.

In [ ]:
# Experiment Step 6: Observe input gradients with PyTorch MultiheadAttention
import torch
import torch.nn as nn
torch.manual_seed(42)

attn_dim = 8
num_heads = 2
attention = nn.MultiheadAttention(
    embed_dim=attn_dim,
    num_heads=num_heads,
    batch_first=True,
)

attn_x = torch.randn(batch_size, seq_len, attn_dim, requires_grad=True)

# True means this position cannot be seen. Upper triangle True = mask the future.
attn_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

attn_out, attn_weights = attention(
    attn_x,
    attn_x,
    attn_x,
    attn_mask=attn_mask,
    need_weights=True,
    average_attn_weights=False,
)

attn_loss = attn_out[:, -1, :].pow(2).mean()
attn_loss.backward()

attn_grad_by_pos = attn_x.grad.norm(dim=-1).squeeze(0).detach()
last_token_weights = attn_weights[0, :, -1, :].mean(dim=0).detach()

print("=== Gradients from last Attention output ===")
print(f"loss from last output: {attn_loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    grad = attn_grad_by_pos[pos].item()
    weight = last_token_weights[pos].item()
    print(f"position {pos:2d}: grad={grad:.10f}, attention_weight={weight:.10f}")

print()
print("From this run's results:")
print(f"  Beginning position gradient: {attn_grad_by_pos[0].item():.3e}")
print(f"  Middle position gradient:    {attn_grad_by_pos[seq_len // 2].item():.3e}")
print(f"  Last position gradient:       {attn_grad_by_pos[-1].item():.3e}")
print(f"  Last token's attention weight sum: {last_token_weights.sum().item():.3f}")
print()
print("Observation: earlier positions no longer rely solely on step-by-step propagation; they can be directly seen by the last position.")

**Putting RNN and Attention on the same plot**

Look at the two curves first, then read the conclusion.

The blue line is RNN: gradients at distant positions easily drop close to 0.

The orange line is Attention: earlier position gradients are typically still smaller than the last position, but they are not forced to propagate through many steps—they connect directly to the last position through attention weights.

In [ ]:
import matplotlib.pyplot as plt
# Comparison: RNN vs Attention gradients backpropagated from the last position to each input position
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), label="RNN")
plt.plot(positions.numpy(), attn_grad_by_pos.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("Gradient from last position: RNN vs Attention")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("From this plot we conclude:")
print("  RNN's distant gradients are more likely to drop close to 0.")
print("  Attention's distant positions may also have smaller gradients, but they have a direct connection to the last position.")
print("  This is one reason Attention is better suited for learning long-range relationships.")

**Where does the last token actually look?**

Attention doesn't just provide gradients; it also produces attention weights showing how much the last position attends to each other position.

These weights are not hand-crafted—they are computed by `nn.MultiheadAttention` during this forward pass.

In [ ]:
import matplotlib.pyplot as plt
# Last token's attention weights across the entire sequence
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), last_token_weights.numpy(), linewidth=2)
plt.scatter(positions.numpy(), last_token_weights.numpy(), s=10)
plt.xlabel("Key position")
plt.ylabel("Attention weight")
plt.title("Last token attention weights")
plt.grid(True, alpha=0.3)
plt.show()

print("From this plot we conclude:")
print("  The last token's attention weight is distributed across multiple positions.")
print("  Positions with larger weights more directly influence the last position's output.")

**One more useful plot: cumulative gradient distribution**

The curves above tell us how large the gradient is at each position.

But there is another question: if we sum up gradients across all positions, how much gradient mass is concentrated near the end?

Below we plot cumulative gradient mass. A curve that rises later means gradient mass is more concentrated toward the end.

In [ ]:
import matplotlib.pyplot as plt
# Insight plot: cumulative gradient distribution
import torch
rnn_mass = grad_by_pos / grad_by_pos.sum()
attn_mass = attn_grad_by_pos / attn_grad_by_pos.sum()

rnn_cumsum = torch.cumsum(rnn_mass, dim=0)
attn_cumsum = torch.cumsum(attn_mass, dim=0)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), rnn_cumsum.numpy(), label="RNN")
plt.plot(positions.numpy(), attn_cumsum.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Cumulative gradient mass")
plt.title("Where does the gradient mass accumulate?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

rnn_first_half = rnn_mass[:seq_len // 2].sum().item()
attn_first_half = attn_mass[:seq_len // 2].sum().item()

print("From the cumulative distribution:")
print(f"  RNN first-half gradient share:       {rnn_first_half:.3%}")
print(f"  Attention first-half gradient share: {attn_first_half:.3%}")
print()
print("Observation: in this run, RNN's gradient mass is more concentrated at the tail.")
print("Attention still favors the tail, but the first half also receives a more visible share of gradient mass.")

**What do these experiments actually show?**

Note that this does not mean every distant token's gradient under Attention will be large. A more accurate statement is:

1. RNN's distant information must pass through many steps, and learning signals tend to grow weaker along the way.
2. Attention lets the last position directly see all earlier positions, shortening the path.
3. Actual gradient magnitudes still depend on parameters, input, and attention weights.

That is why we later study Q/K/V, masks, and multi-head: they determine exactly how "direct seeing" works in practice.

## Exercises

> You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

**Exercise 1: Attention score**

The first step of Attention is computing relevance scores using query and key.

**Hint**: The dot product of two vectors can be computed with `(q * k).sum()`.

In [ ]:
# Exercise 1: Attention score fill-in-the-blank
import torch

q = torch.tensor([1.0, 2.0, 0.0])
k = torch.tensor([3.0, 1.0, 4.0])

# TODO: Replace the triple-quoted content below with your code
score = """Compute the dot product of q and k here"""

assert not isinstance(score, str), "Please replace the triple-quoted placeholder first"
assert score.item() == 5.0, score
print("Exercise 1 passed: you remembered that the attention score is the QK dot product")

**Exercise 2: Causal mask**

During GPT generation, future tokens cannot be peeked at, so position `i` can only see positions `0...i`.

**Hint**: `torch.tril(torch.ones(seq_len, seq_len))` generates a lower triangular matrix.

In [ ]:
# Exercise 2: Causal Mask fill-in-the-blank
import torch

seq_len = 5

# TODO: Generate a 5x5 lower triangular causal mask
mask = """Generate the causal mask here"""

expected = torch.tensor([
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
])
assert not isinstance(mask, str), "Please replace the triple-quoted placeholder first"
assert torch.equal(mask, expected), mask
print("Exercise 2 passed: you understand why GPT can only look at the past")

## References

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — the original Transformer paper; definitions of Scaled Dot-Product Attention and Multi-Head Attention are from this paper
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — a line-by-line implementation and annotation of the original paper; the variance argument for the scaling factor \u221ad_k in this notebook references the derivation from this article
- Karpathy, [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) — a video tutorial building GPT from scratch in code